# rotation-matrix-3d-y-axis — ex10: SLERP between two Y-rotations vs matrix-LERP — orthogonality gap

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d-y-axis`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rotation-matrix-3d-y-axis`** (exercise 10). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d-y-axis"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## SLERP on Y-axis rotations — quick refresher

For a single-axis rotation family `R(θ) = R_y(θ)`, spherical linear interpolation (SLERP) between `R(α)` and `R(β)` collapses to **linear interpolation of the angle**: `R_slerp(s) = R(α + s·(β-α))` for `s ∈ [0, 1]`. The exotic quaternion machinery isn't needed when the rotation axis is fixed.

**Why the path stays on SO(3).** Every `R_y(θ)` is orthogonal (`R^T R = I`) and has `det(R) = 1`. Linear interpolation of the angle keeps you within the family; the resulting matrices remain on the rotation manifold. Compare this with **linear interpolation of the matrices** `(1-s)·R(α) + s·R(β)`, which does NOT stay on SO(3) — the midpoint is closer to a scaled rotation.

**This drill (ex10) vs ex1-9.** Earlier exercises built R, applied R to vectors, composed Rx·Ry·Rz, ran angle sweeps, and verified the inverse identity. ex10 walks a *path* on SO(3) and quantifies the gap between angle-SLERP and matrix-LERP at every step.

### Exercise 10 — SLERP between two Y-rotations vs matrix-LERP — orthogonality gap

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the deviation from SO(3) along two interpolation paths between Y-rotations — angle-SLERP (stays on manifold) vs matrix-LERP (leaves manifold) — by quantifying `||R^T R - I||_F` per step.
> Keywords: SLERP, manifold, orthogonality, interpolation
> ```

**KCs targeted:** `ry-orthogonality`, `ry-angle-interpolation`

Implement `ex10_slerp_vs_lerp(alpha, beta, n_steps)`.

Walk two interpolation paths from `R_y(alpha)` to `R_y(beta)` using `n_steps` (inclusive of both endpoints) values of `s ∈ [0, 1]`:

1. **Angle-SLERP path.** At each `s`, build `R_slerp = R_y(alpha + s·(beta - alpha))` from scratch using `cos/sin`. This stays on the manifold by construction.
2. **Matrix-LERP path.** At each `s`, build `R_lerp = (1 - s) · R_y(alpha) + s · R_y(beta)`. This linearly blends the matrices — off-manifold at every interior step.
3. For each path and each step, compute the Frobenius-norm gap from orthogonality: `gap = ||R^T R - I||_F`.

Return `(slerp_gaps, lerp_gaps)` — two 1-D tensors of length `n_steps`, both `float32`.

Use `torch.linspace(0, 1, n_steps)` for `s`. SLERP gaps should be ≈ 0 at every step (floating-point noise only); LERP gaps should be 0 at the two endpoints and reach a maximum near the midpoint.

The visualization overlays both gap curves so you can see the off-manifold bulge of matrix-LERP versus the flat SLERP baseline.

In [ ]:
def ex10_slerp_vs_lerp(
    alpha: float,
    beta: float,
    n_steps: int,
) -> tuple[Tensor, Tensor]:
    def Ry(theta: Tensor) -> Tensor:
        c, s = t.cos(theta), t.sin(theta)
        z, o = t.zeros_like(c), t.ones_like(c)
        return t.stack([
            t.stack([ c, z, s]),
            t.stack([ z, o, z]),
            t.stack([-s, z, c]),
        ])

    a = t.tensor(alpha, dtype=t.float32)
    b = t.tensor(beta, dtype=t.float32)
    Ra, Rb = Ry(a), Ry(b)
    eye = t.eye(3)
    ss = t.linspace(0.0, 1.0, n_steps)
    slerp_gaps = t.empty(n_steps, dtype=t.float32)
    lerp_gaps = t.empty(n_steps, dtype=t.float32)
    for i, s in enumerate(ss):
        Rs = Ry(a + s * (b - a))
        Rl = (1 - s) * Ra + s * Rb
        slerp_gaps[i] = (Rs.T @ Rs - eye).norm()
        lerp_gaps[i] = (Rl.T @ Rl - eye).norm()
    return slerp_gaps, lerp_gaps


<details><summary>Solution</summary>

```python
def ex10_slerp_vs_lerp(
    alpha: float,
    beta: float,
    n_steps: int,
) -> tuple[Tensor, Tensor]:
    def Ry(theta: Tensor) -> Tensor:
        c, s = t.cos(theta), t.sin(theta)
        z, o = t.zeros_like(c), t.ones_like(c)
        return t.stack([
            t.stack([ c, z, s]),
            t.stack([ z, o, z]),
            t.stack([-s, z, c]),
        ])

    a = t.tensor(alpha, dtype=t.float32)
    b = t.tensor(beta, dtype=t.float32)
    Ra, Rb = Ry(a), Ry(b)
    eye = t.eye(3)
    ss = t.linspace(0.0, 1.0, n_steps)
    slerp_gaps = t.empty(n_steps, dtype=t.float32)
    lerp_gaps = t.empty(n_steps, dtype=t.float32)
    for i, s in enumerate(ss):
        Rs = Ry(a + s * (b - a))
        Rl = (1 - s) * Ra + s * Rb
        slerp_gaps[i] = (Rs.T @ Rs - eye).norm()
        lerp_gaps[i] = (Rl.T @ Rl - eye).norm()
    return slerp_gaps, lerp_gaps
```

**Why angle-SLERP works for single-axis rotations.** SO(2) (rotations in a plane) is a 1-D Lie group parameterized by the angle. Y-rotations form an isomorphic subgroup of SO(3): the angle composes additively, so linear interpolation of the angle IS the geodesic. For multi-axis SLERP you'd need quaternions or a matrix exponential — but here the cheap path is the right path.

**Why matrix-LERP fails.** Linear blends of rotation matrices are no longer rotations: `(1-s)R_a + s R_b` is a scaled, sheared matrix whose columns are NOT unit-orthogonal. The Frobenius gap `||R^T R - I||_F` peaks at the midpoint and is symmetric about `s=0.5` because the geometry of the path is symmetric.

**Practical takeaway.** If you ever need to morph between two orientations (camera path, joint interpolation, latent-space rotation), interpolate the PARAMETER, not the matrix. The whole point of choosing a parameterization is that the manifold is closed under the parameter's linear structure.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex10',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()